In [1]:
from dotenv import load_dotenv
print(load_dotenv())


True


In [17]:
from langchain_community.document_loaders import WebBaseLoader
url = "https://docs.langchain.com/langsmith/billing"
loader = WebBaseLoader(url)
documents = loader.load()


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embedded_docs = embeddings.embed_documents([doc.page_content for doc in chunks])
# retriever = embedded_docs.as_retriever()
# retriever
embedded_docs

In [24]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)


In [ ]:
vectorstore.similarity_search("How to manage billing in my account in LangSmith?", k=2)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant that answers questions about LangSmith. Use the following context to answer the question at the end. If you don't know the answer, just say you don't know, don't try to make up an answer.\n\nContext: {context}\nQuestion: {question}"
)
llm = ChatOpenAI(model="gpt-4.1-nano")
query = "How to manage billing in my account in LangSmith?"
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke(query))

OpenAIPermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_pdMsqL91WBqmlvaWNMfjY4W8` does not have access to model `gpt-4o-mini`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}